<a href="https://colab.research.google.com/github/mani9k9/Machine-Learning/blob/main/Worldquant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step1**

In [19]:
import pandas as pd
import random

# ===============================
# LOAD DATA
# ===============================

base_url = "https://docs.google.com/spreadsheets/d/137K0yl2iITHZiaaBfzWkOxIceUyFjwUJHCJLjFNB0Cs/gviz/tq?tqx=out:csv&sheet="

op_df = pd.read_csv(base_url + "Operators")
field_df = pd.read_csv(base_url + "fields")

OPS = op_df.iloc[:, 0].dropna().tolist()
FIELDS = field_df.iloc[:, 0].dropna().tolist()

if len(OPS) == 0:
    OPS = ["ts_delta", "ts_mean", "ts_rank", "ts_corr"]

if len(FIELDS) == 0:
    FIELDS = ["close", "open", "high", "low", "volume"]

# ===============================
# BASIC BUILDING BLOCKS
# ===============================

def f():
    return random.choice(FIELDS)

def op():
    return random.choice(OPS)

# ===============================
# ALPHA LAYERS
# ===============================

def layer_1():
    return f"{op()}({f()},{random.choice([5,10,20,30])})"

def layer_2():
    return f"rank({op()}({f()},{f()},{random.choice([5,10,20])}))"

def layer_3():
    a = layer_1()
    b = layer_2()
    return f"rank({a} + {b})"

def layer_4():
    a = layer_3()
    b = layer_1()
    return f"rank({a} / ({b} + 0.000001))"

# ===============================
# GENERATOR
# ===============================

def generate_alpha():
    choice = random.randint(1, 4)

    if choice == 1:
        return layer_1()
    elif choice == 2:
        return layer_2()
    elif choice == 3:
        return layer_3()
    else:
        return layer_4()

# ===============================
# TEST RUN
# ===============================

generated = [generate_alpha() for _ in range(30)]
generated = list(set(generated))

print("\n🔥 GENERATED ALPHAS (STEP 1)\n")

for i, a in enumerate(generated):
    print(f"{i+1}. {a}")


🔥 GENERATED ALPHAS (STEP 1)

1. rank(ts_rank(x, d)(anl14_actvalue_bvps_fp0,5) + rank(ts_sum(x, d)(anl14_actvalue_nav_fy0,anl14_actvalue_capex_fy0,5)))
2. rank(ts_arg_max(x, d)(anl14_actvalue_nav_fy0,5) + rank(ts_delta(x, d)(anl14_actvalue_ndebt_fy0,anl14_actvalue_bvps_fp0,20)))
3. rank(sign(x)(anl14_actvalue_div_fp0,10) + rank(ts_rank(x, d)(anl14_actvalue_div_fp0,anl14_actvalue_cfps_fp0,10)))
4. rank(rank(ts_max(x, d)(anl10_cpsff_2371,20) + rank(ts_sum(x, d)(anl14_actvalue_capex_fp0,anl10_cpsfq1_pred_surps_v2_2364,20))) / (abs(x)(anl14_actvalue_capex_fp0,10) + 0.000001))
5. rank(ts_rank(x, d)(anl14_actvalue_capex_fy0,anl14_actvalue_ntp_fp0,10))
6. delay(x, d)(anl14_actvalue_capex_fy0,10)
7. add(x, y, ..., filter=false)(anl14_actvalue_bvps_fy0,20)
8. rank(ts_delta(x, d)(anl10_cpsfq1_pred_surps_v2_2364,anl14_actvalue_capex_fp0,5))
9. rank(sqrt(x)(anl14_actvalue_div_fp0,anl10_cpsfq1_smart_ests_v0_2372,5))
10. rank(log(x)(anl14_actvalue_ndebt_fy0,20) + rank(clip(x, min, max)(anl10_cpsfq1_

# **Step 2**

In [20]:
import re

# ===============================
# SAFE VALIDATION RULES
# ===============================

def is_valid_alpha(alpha: str):

    if not isinstance(alpha, str):
        return False

    a = alpha.strip()

    # ❌ too short
    if len(a) < 8:
        return False

    # ❌ must have brackets (all valid alphas are function-based)
    if "(" not in a or ")" not in a:
        return False

    # ❌ must have at least one field OR ts operator
    valid_tokens = ["close", "open", "high", "low", "volume", "ts_", "rank"]
    if not any(t in a for t in valid_tokens):
        return False

    # ❌ bracket mismatch
    if a.count("(") != a.count(")"):
        return False

    # ❌ avoid broken function calls like ",)"
    if re.search(r",\s*\)", a):
        return False

    # ❌ reject empty operators like ()
    if "()" in a:
        return False

    return True

# ===============================
# FILTER FUNCTION
# ===============================

def filter_alphas(alpha_list):

    if alpha_list is None or len(alpha_list) == 0:
        return [], []

    valid = []
    rejected = []

    for a in alpha_list:
        if is_valid_alpha(a):
            valid.append(a)
        else:
            rejected.append(a)

    return valid, rejected

# ===============================
# RUN STEP 2
# ===============================

valid_alphas, rejected_alphas = filter_alphas(generated)

print("\n🔥 STEP 2 RESULTS (FIXED)\n")

print("Total Generated:", len(generated))
print("Valid Alphas   :", len(valid_alphas))
print("Rejected       :", len(rejected_alphas))

print("\n✅ SAMPLE VALID ALPHAS:\n")

for i, a in enumerate(valid_alphas[:20]):
    print(i+1, a)


🔥 STEP 2 RESULTS (FIXED)

Total Generated: 30
Valid Alphas   : 26
Rejected       : 4

✅ SAMPLE VALID ALPHAS:

1 rank(ts_rank(x, d)(anl14_actvalue_bvps_fp0,5) + rank(ts_sum(x, d)(anl14_actvalue_nav_fy0,anl14_actvalue_capex_fy0,5)))
2 rank(ts_arg_max(x, d)(anl14_actvalue_nav_fy0,5) + rank(ts_delta(x, d)(anl14_actvalue_ndebt_fy0,anl14_actvalue_bvps_fp0,20)))
3 rank(sign(x)(anl14_actvalue_div_fp0,10) + rank(ts_rank(x, d)(anl14_actvalue_div_fp0,anl14_actvalue_cfps_fp0,10)))
4 rank(rank(ts_max(x, d)(anl10_cpsff_2371,20) + rank(ts_sum(x, d)(anl14_actvalue_capex_fp0,anl10_cpsfq1_pred_surps_v2_2364,20))) / (abs(x)(anl14_actvalue_capex_fp0,10) + 0.000001))
5 rank(ts_rank(x, d)(anl14_actvalue_capex_fy0,anl14_actvalue_ntp_fp0,10))
6 rank(ts_delta(x, d)(anl10_cpsfq1_pred_surps_v2_2364,anl14_actvalue_capex_fp0,5))
7 rank(sqrt(x)(anl14_actvalue_div_fp0,anl10_cpsfq1_smart_ests_v0_2372,5))
8 rank(log(x)(anl14_actvalue_ndebt_fy0,20) + rank(clip(x, min, max)(anl10_cpsfq1_pred_surps_v1_2381,anl10_cpsfq1_

# **Step 3**

In [21]:
import numpy as np
import pandas as pd

# ===============================
# SAFE FEATURE EXTRACTION
# ===============================

def extract_features(alpha: str):

    alpha = str(alpha)

    return {
        "length": len(alpha),
        "ts_count": alpha.count("ts_"),
        "rank_count": alpha.count("rank"),
        "op_count": alpha.count("+") + alpha.count("-") + alpha.count("*") + alpha.count("/"),
        "field_close": alpha.count("close"),
        "field_volume": alpha.count("volume"),
        "nesting": alpha.count("(") - alpha.count(")")
    }

# ===============================
# SAFE SCORER
# ===============================

def score_alpha(alpha: str):

    f = extract_features(alpha)

    return (
        0.25 * f["ts_count"] +
        0.20 * f["rank_count"] +
        0.15 * f["op_count"] +
        0.10 * f["nesting"] +
        0.15 * (f["field_close"] + f["field_volume"]) +
        0.15 * np.log1p(f["length"])
    )

# ===============================
# SAFE POPULATION SCORING
# ===============================

def score_population(alpha_list):

    if alpha_list is None or len(alpha_list) == 0:
        print("⚠️ No alphas found. Run STEP 1 + STEP 2 first.")
        return pd.DataFrame(columns=["alpha", "score"])

    results = []

    for a in alpha_list:
        try:
            results.append({
                "alpha": a,
                "score": score_alpha(a)
            })
        except:
            continue

    if len(results) == 0:
        print("⚠️ No valid scored alphas generated.")
        return pd.DataFrame(columns=["alpha", "score"])

    df = pd.DataFrame(results)
    return df.sort_values("score", ascending=False)

# ===============================
# RUN
# ===============================

try:
    scored_df = score_population(valid_alphas)
except Exception as e:
    print("❌ Error:", e)
    scored_df = pd.DataFrame()

# ===============================
# OUTPUT
# ===============================

print("\n🔥 STEP 3 OUTPUT\n")

if len(scored_df) == 0:
    print("⚠️ No scored results available")
else:
    for _, r in scored_df.head(20).iterrows():
        print("=" * 80)
        print(r["alpha"])
        print("Score:", round(r["score"], 4))


🔥 STEP 3 OUTPUT

rank(rank(ts_max(x, d)(anl10_cpsff_2371,20) + rank(ts_sum(x, d)(anl14_actvalue_capex_fp0,anl10_cpsfq1_pred_surps_v2_2364,20))) / (abs(x)(anl14_actvalue_capex_fp0,10) + 0.000001))
Score: 2.3281
rank(ts_rank(x, d)(anl14_actvalue_bvps_fp0,5) + rank(ts_sum(x, d)(anl14_actvalue_nav_fy0,anl14_actvalue_capex_fy0,5)))
Score: 1.9669
rank(rank(multiply(x, y, ..., filter=false)(anl14_actvalue_capex_fy0,30) + rank(sign(x)(anl14_actvalue_ndebt_fy0,anl14_actvalue_nav_fy0,5))) / (group_sum(x, group)(anl14_actvalue_bvps_fp0,5) + 0.000001))
Score: 1.8477
rank(rank(group_mean(x, group)(anl14_actvalue_bvps_fy0,5) + rank(power(x, y)(anl10_cpsff_2371,anl14_actvalue_cfps_fp0,20))) / (max(x, y, ...)(anl10_cpsfq1_pred_surps_v1_2381,30) + 0.000001))
Score: 1.8378
rank(ts_corr(x, y, d)(anl14_actvalue_nav_fy0,20) + rank(ts_mean(x, d)(anl10_cpsfq1_pred_surps_v0_2389,anl14_actvalue_div_fp0,5)))
Score: 1.7801
rank(abs(x)(anl14_actvalue_cfps_fp0,5) + rank(ts_sum(x, d)(anl10_cpsfq1_pred_surps_v0_238

# **STEP 4 — DIVERSITY ENGINE**

In [22]:
import re
from collections import defaultdict

# ===============================
# STRUCTURE NORMALIZER
# ===============================

def normalize(alpha: str):

    a = str(alpha)

    # replace fields with generic token
    a = re.sub(r"\b(close|open|high|low|volume)\b", "X", a)

    # replace numbers with N
    a = re.sub(r"\d+", "N", a)

    # remove extra spaces
    a = re.sub(r"\s+", "", a)

    return a

# ===============================
# SIMILARITY CLUSTERING
# ===============================

def filter_diverse(alphas, max_per_cluster=1):

    clusters = defaultdict(list)

    for a in alphas:
        key = normalize(a)
        clusters[key].append(a)

    diverse = []

    for k, items in clusters.items():
        # keep only best representative (first one for now)
        diverse.extend(items[:max_per_cluster])

    return diverse, clusters

# ===============================
# RUN DIVERSITY FILTER
# ===============================

try:
    diverse_alphas, clusters = filter_diverse(valid_alphas)
except NameError:
    print("⚠️ Run STEP 2 first")
    diverse_alphas, clusters = [], {}

# ===============================
# OUTPUT
# ===============================

print("\n🔥 STEP 4 RESULTS (DIVERSITY FILTER)\n")

print("Before:", len(valid_alphas) if 'valid_alphas' in globals() else 0)
print("After :", len(diverse_alphas))

print("\n✅ DIVERSE ALPHAS SAMPLE:\n")

for i, a in enumerate(diverse_alphas[:20]):
    print(i+1, a)


🔥 STEP 4 RESULTS (DIVERSITY FILTER)

Before: 26
After : 26

✅ DIVERSE ALPHAS SAMPLE:

1 rank(ts_rank(x, d)(anl14_actvalue_bvps_fp0,5) + rank(ts_sum(x, d)(anl14_actvalue_nav_fy0,anl14_actvalue_capex_fy0,5)))
2 rank(ts_arg_max(x, d)(anl14_actvalue_nav_fy0,5) + rank(ts_delta(x, d)(anl14_actvalue_ndebt_fy0,anl14_actvalue_bvps_fp0,20)))
3 rank(sign(x)(anl14_actvalue_div_fp0,10) + rank(ts_rank(x, d)(anl14_actvalue_div_fp0,anl14_actvalue_cfps_fp0,10)))
4 rank(rank(ts_max(x, d)(anl10_cpsff_2371,20) + rank(ts_sum(x, d)(anl14_actvalue_capex_fp0,anl10_cpsfq1_pred_surps_v2_2364,20))) / (abs(x)(anl14_actvalue_capex_fp0,10) + 0.000001))
5 rank(ts_rank(x, d)(anl14_actvalue_capex_fy0,anl14_actvalue_ntp_fp0,10))
6 rank(ts_delta(x, d)(anl10_cpsfq1_pred_surps_v2_2364,anl14_actvalue_capex_fp0,5))
7 rank(sqrt(x)(anl14_actvalue_div_fp0,anl10_cpsfq1_smart_ests_v0_2372,5))
8 rank(log(x)(anl14_actvalue_ndebt_fy0,20) + rank(clip(x, min, max)(anl10_cpsfq1_pred_surps_v1_2381,anl10_cpsfq1_pred_surps_v0_2389,5)))


# **STEP 5 — EVOLUTION ENGINE**

In [23]:
import pandas as pd
import random
import re
from collections import defaultdict

# =================================================
# LOAD OP + FIELD
# =================================================

base_url = "https://docs.google.com/spreadsheets/d/137K0yl2iITHZiaaBfzWkOxIceUyFjwUJHCJLjFNB0Cs/gviz/tq?tqx=out:csv&sheet="

op_df = pd.read_csv(base_url + "Operators")
field_df = pd.read_csv(base_url + "fields")

OPS = op_df.iloc[:, 0].dropna().tolist()
FIELDS = field_df.iloc[:, 0].dropna().tolist()

if len(OPS) == 0:
    OPS = ["ts_delta", "ts_mean", "ts_rank", "ts_corr"]

if len(FIELDS) == 0:
    FIELDS = ["close", "open", "high", "low", "volume"]

# =================================================
# STEP 1: GENERATOR
# =================================================

def f(): return random.choice(FIELDS)
def op(): return random.choice(OPS)

def generate_alpha():
    return random.choice([
        f"rank(ts_delta({f()},10))",
        f"rank(ts_mean({f()},20))",
        f"rank(ts_corr({f()},{f()},20))",
        f"rank({op()}({f()},{f()},20))",
        f"rank(ts_rank({f()},30))"
    ])

# =================================================
# STEP 2: VALIDATOR
# =================================================

def is_valid(alpha):

    if not isinstance(alpha, str):
        return False

    a = alpha.strip()

    if len(a) < 8:
        return False

    if "(" not in a or ")" not in a:
        return False

    if a.count("(") != a.count(")"):
        return False

    if "()" in a:
        return False

    return True

# =================================================
# STEP 3: SCORER (STRUCTURAL IC PROXY)
# =================================================

def score(alpha):

    return (
        alpha.count("ts_") * 0.25 +
        alpha.count("rank") * 0.20 +
        (alpha.count("+") + alpha.count("-")) * 0.15 +
        len(alpha) * 0.01
    )

# =================================================
# STEP 4: DIVERSITY ENGINE
# =================================================

def normalize(a):
    a = re.sub(r"\b(close|open|high|low|volume)\b", "X", a)
    a = re.sub(r"\d+", "N", a)
    a = re.sub(r"\s+", "", a)
    return a

def diversify(alphas):

    clusters = defaultdict(list)

    for a in alphas:
        clusters[normalize(a)].append(a)

    return [items[0] for items in clusters.values()]

# =================================================
# EVOLUTION LOOP
# =================================================

GENERATIONS = 5
POP_SIZE = 100
TOP_K = 20

seed_pool = []

print("\n🔥 STARTING ALPHA EVOLUTION ENGINE\n")

for g in range(GENERATIONS):

    # ----------------------------
    # GENERATE POPULATION
    # ----------------------------
    population = []

    for _ in range(POP_SIZE):
        a = generate_alpha()

        if not is_valid(a):
            continue

        population.append(a)

    # ----------------------------
    # SCORE POPULATION
    # ----------------------------
    scored = [(a, score(a)) for a in population]
    scored = sorted(scored, key=lambda x: x[1], reverse=True)

    top = [a for a, s in scored[:TOP_K]]

    # ----------------------------
    # DIVERSITY FILTER
    # ----------------------------
    top = diversify(top)

    # ----------------------------
    # SEED NEXT GENERATION
    # ----------------------------
    seed_pool.extend(top)

    print(f"\nGEN {g}")
    print("Top score:", scored[0][1] if scored else 0)
    print("Diverse elites:", len(top))

    # keep memory bounded
    seed_pool = seed_pool[-200:]

# =================================================
# FINAL OUTPUT
# =================================================

print("\n🔥 FINAL EVOLVED ALPHAS:\n")

for i, a in enumerate(seed_pool[:30]):
    print(f"{i+1}. {a}")


🔥 STARTING ALPHA EVOLUTION ENGINE


GEN 0
Top score: 1.5699999999999998
Diverse elites: 15

GEN 1
Top score: 1.54
Diverse elites: 15

GEN 2
Top score: 1.51
Diverse elites: 18

GEN 3
Top score: 1.53
Diverse elites: 16

GEN 4
Top score: 1.42
Diverse elites: 17

🔥 FINAL EVOLVED ALPHAS:

1. rank(ts_covariance(x, y, d)(anl14_actvalue_div_fp0,anl10_cpsfq1_smart_ests_v0_2372,20))
2. rank(signed_power(x, y)(anl14_actvalue_capex_fy0,anl10_cpsfq1_smart_ests_v0_2372,20))
3. rank(ts_corr(anl10_cpsfq1_pred_surps_v1_2381,anl10_cpsfq1_pred_surps_v1_2381,20))
4. rank(ts_std(x, d)(anl14_actvalue_nav_fy0,anl10_cpsfq1_pred_surps_v0_2389,20))
5. rank(ts_delta(anl10_cpsfq1_smart_ests_v0_2372,10))
6. rank(ts_arg_min(x, d)(anl14_actvalue_capex_fp0,anl14_actvalue_bvps_fy0,20))
7. rank(ts_arg_min(x, d)(anl14_actvalue_bvps_fp0,anl14_actvalue_bvps_fy0,20))
8. rank(ts_corr(anl14_actvalue_capex_fy0,anl10_cpsfq1_pred_surps_v1_2381,20))
9. rank(ts_mean(anl10_cpsfq1_smart_ests_v0_2372,20))
10. rank(ts_corr(anl10_cps